# 🚀 Stock Bot AutoEncoder Training on Google Colab

이 노트북은 한국 주식 시장 스캘핑을 위한 AutoEncoder 기반 임베딩 모델을 Google Colab에서 훈련합니다.

## 📋 Overview
- **목적**: 시계열 데이터를 128차원 임베딩으로 압축
- **모델**: Masked AutoEncoder with Transformer
- **데이터**: 전처리된 HDF5 배치 파일
- **훈련 시간**: 약 4-6시간 (GPU 사용시)

## 🔧 Environment Setup

In [ ]:
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install h5py numpy pandas tqdm matplotlib seaborn

# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Setup directories
import os
os.chdir('/content')
!mkdir -p models logs data

# Data path
DATA_PATH = '/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/pre_training_data'

# Check data
if os.path.exists(DATA_PATH):
    print(f"✅ Data found at: {DATA_PATH}")
    months = [d for d in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, d))]
    print(f"Available months: {sorted(months)}")
else:
    print(f"❌ Data not found at: {DATA_PATH}")

## Github

In [ ]:
# GitHub 저장소 클론 (프로젝트 코드)
import os
from google.colab import userdata

# 🔑 아이콘 클릭 → Add new secret
# Name: GITHUB_TOKEN
# Value: your_personal_access_token
try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 Using GitHub token for authentication")
except:
    print("⚠️ GITHUB_TOKEN not found. Using public clone (may have rate limits)")
    use_token = False

repo_path = '/content/stock-bot2'

# 이미 클론되어 있으면 스킵
if not os.path.exists(repo_path):
    print("📥 Cloning repository...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git {repo_path}
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git {repo_path}

    if os.path.exists(repo_path):
        print("✅ Repository cloned successfully!")
    else:
        print("❌ Repository cloning failed. Please check your GitHub token and repository access permissions.")
        # Early exit if cloning failed to prevent further errors
        # No need to change directory or sys path if clone failed
        print(f"📂 Current directory: {os.getcwd()}")
        sys.exit("Repository cloning failed.") # Stop execution here.

else:
    print("📁 Repository already exists, updating...")
    %cd {repo_path}
    !git fetch origin && git pull
    print("✅ Repository updated!")

# 작업 디렉토리 변경 및 Python path 추가 (성공적으로 클론되거나 업데이트된 경우에만)
%cd {repo_path}
import sys
# Ensure the project root is in the Python path
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("✅ Repository ready!")
print(f"📂 Current directory: {os.getcwd()}")


## 📚 Import Libraries and Models

In [ ]:
import sys
# Ensure the project root is in the Python path
if '/content/stock-bot2' not in sys.path:
    sys.path.insert(0, '/content/stock-bot2')

# Import standard libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import json
import random
import math
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import project modules
try:
    # Attempt to directly import the modules and their classes
    from ai_trader.embedding.autoencoder_model import MaskedAutoEncoder
    from ai_trader.embedding.data import PreprocessedDataset, BatchCollator
    print("✅ Successfully imported project modules")

except ImportError as e:
    print(f"❌ Failed to import project modules directly: {e}")
    print("📝 Attempting to execute files and check definitions...")
    try:
        # Execute the files to define classes/functions in the current scope
        exec(open('/content/stock-bot2/ai_trader/embedding/autoencoder_model.py').read())
        exec(open('/content/stock-bot2/ai_trader/embedding/data.py').read())

        # Check if the expected classes are defined in the current global scope
        if 'MaskedAutoEncoder' in globals() and 'PreprocessedDataset' in globals() and 'BatchCollator' in globals():
             print("✅ Classes seem to be defined in the global scope after exec.")
             # Assign the classes to the global scope if they are defined by exec
             MaskedAutoEncoder = globals()['MaskedAutoEncoder']
             PreprocessedDataset = globals()['PreprocessedDataset']
             BatchCollator = globals()['BatchCollator']
             print("✅ Successfully made project classes available via exec and globals check.")
        else:
             print("❌ Expected classes not found in global scope after exec.")
             print("📝 Please ensure the files define the classes/functions correctly.")
             print(f"Defined in globals after exec: {list(globals().keys())}") # Debugging line

    except Exception as exec_error:
        print(f"❌ An error occurred during file execution or class availability check: {exec_error}")
        print("📝 Please ensure the file paths are correct and the files are valid Python scripts.")

In [ ]:
import sys
# Ensure the project root is in the Python path
if '/content/stock-bot2' not in sys.path:
    sys.path.insert(0, '/content/stock-bot2')

# Import standard libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import h5py
import json
import random
import math
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import time
from datetime import datetime

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Import project modules with fallback
try:
    from ai_trader.embedding.autoencoder_model import MaskedAutoEncoder
    print("✅ Successfully imported MaskedAutoEncoder")
except ImportError as e:
    print(f"❌ Failed to import MaskedAutoEncoder: {e}")
    print("📝 Loading model from file...")
    # This block executes the file to define the class if direct import fails.
    # The assumption here is that the file path is correct.
    exec(open('/content/stock-bot2/ai_trader/embedding/autoencoder_model.py').read())

# Define data classes for HDF5 batch files (Fixed version)
class PreprocessedDataset(Dataset):
    def __init__(self, data_dir, months, max_batches_per_month=None, configured_seq_len=None): # Added configured_seq_len
        self.data_dir = Path(data_dir)
        self.batch_files = []
        self.num_features = 28 # Initialize with a default, will be updated by _inspect_data_structure
        self._actual_h5_seq_len = 60 # Internal variable to store the actual sequence length found in HDF5 files

        # The 'seq_len' attribute of the dataset instance will now represent the *target* length (e.g., 120)
        self.seq_len = configured_seq_len if configured_seq_len is not None else self._actual_h5_seq_len # 목표 시퀀스 길이 설정

        for month in months:
            month_dir = self.data_dir / month
            if not month_dir.exists():
                continue

            batch_info_file = month_dir / 'batch_info.json'
            if batch_info_file.exists():
                with open(batch_info_file, 'r') as f:
                    info = json.load(f)
                    num_batches = info['num_batches']
                    if max_batches_per_month:
                        num_batches = min(num_batches, max_batches_per_month)

                    for batch_idx in range(num_batches):
                        batch_file = month_dir / f'batch_{batch_idx:06d}.h5'
                        if batch_file.exists():
                            self.batch_files.append(str(batch_file))

        random.shuffle(self.batch_files)

        if self.batch_files:
            self._inspect_data_structure() # This will update self._actual_h5_seq_len and self.num_features

        print(f"Found {len(self.batch_files)} batch files")
        print(f"Data shape (target): seq_len={self.seq_len}, num_features={self.num_features}")
        if self._actual_h5_seq_len != self.seq_len:
            print(f"⚠️ Warning: Actual sequence length in files is {self._actual_h5_seq_len}, but data will be processed to {self.seq_len} (padded/truncated).")

    def _inspect_data_structure(self):
        """데이터 구조 검사 및 설정"""
        print(f"🔍 Inspecting data structure in: {self.batch_files[0]}")

        with h5py.File(self.batch_files[0], 'r') as f:
            print(f"📁 Available keys in HDF5: {list(f.keys())}")

            if 'sequences' in f:
                sequences = f['sequences']
                print(f"📊 Sequences dataset shape: {sequences.shape}")
                print(f"📊 Sequences dtype: {sequences.dtype}")

                # 실제 데이터 샘플 확인
                if sequences.size > 0:
                    sample = sequences[0] if len(sequences) > 0 else sequences
                    print(f"📋 First sample shape: {sample.shape}")
                    print(f"📋 First sample type: {type(sample)}")

                    if hasattr(sample, 'shape'):
                        if len(sample.shape) == 2:
                            # (seq_len, num_features)
                            self._actual_h5_seq_len = sample.shape[0] # Stores actual length from data
                            self.num_features = sample.shape[1]
                            print(f"✅ 2D sample detected: ({self._actual_h5_seq_len}, {self.num_features})")
                        elif len(sample.shape) == 1:
                            # Flattened sequence, assume a default sequence length for inspection
                            total_features = sample.shape[0]
                            self._actual_h5_seq_len = 60 # Default for inspection if 1D
                            self.num_features = total_features // self._actual_h5_seq_len
                            print(f"🔄 1D sample detected, assuming actual_h5_seq_len=60: ({self._actual_h5_seq_len}, {self.num_features})")
                        else:
                            print(f"⚠️ Unexpected sample shape: {sample.shape}")
                            self._actual_h5_seq_len = 60
                            self.num_features = 28
                    else:
                        print(f"⚠️ Sample is not array-like: {sample}")
                        self._actual_h5_seq_len = 60
                        self.num_features = 28

                # 전체 데이터셋 형태 확인
                if len(sequences.shape) == 3:
                    # (num_sequences, seq_len, num_features)
                    self._actual_h5_seq_len = sequences.shape[1]
                    self.num_features = sequences.shape[2]
                    print(f"✅ 3D dataset detected: {sequences.shape}")
                elif len(sequences.shape) == 2:
                    # (num_sequences, flattened_features) 또는 (seq_len, num_features)
                    if sequences.shape[0] > sequences.shape[1]:
                        # 많은 시퀀스, 적은 피처 -> (num_sequences, flattened)
                        total_features = sequences.shape[1]
                        self._actual_h5_seq_len = 60  # Default for inspection if 2D (many sequences)
                        self.num_features = total_features // self._actual_h5_seq_len
                        print(f"🔄 2D dataset (many sequences): reshaping to ({self._actual_h5_seq_len}, {self.num_features})")
                    else:
                        # 적은 시퀀스, 많은 피처 -> (seq_len, num_features)
                        self._actual_h5_seq_len = sequences.shape[0]
                        self.num_features = sequences.shape[1]
                        print(f"✅ 2D dataset (single sequence): ({self._actual_h5_seq_len}, {self.num_features})")
                else:
                    print(f"⚠️ Unexpected dataset shape: {sequences.shape}")
                    self._actual_h5_seq_len = 60
                    self.num_features = 28

            else:
                print(f"❌ No 'sequences' key found. Available keys: {list(f.keys())}")
                # 기본값 설정
                self._actual_h5_seq_len = 60
                self.num_features = 28

    def __len__(self):
        return len(self.batch_files)

    def __getitem__(self, idx):
        batch_file = self.batch_files[idx]

        with h5py.File(batch_file, 'r') as f:
            if 'sequences' in f:
                sequences = f['sequences'][:] # 시퀀스 데이터 로드
            else:
                # sequences 키가 없으면 다른 키 시도
                keys = list(f.keys())
                print(f"⚠️ 'sequences' not found, trying other keys: {keys}")
                sequences = f[keys[0]][:]  # 첫 번째 키 사용

            # 데이터 형태에 따라 처리
            # Use self._actual_h5_seq_len for reshaping from 1D/2D to 3D correctly based on original data
            if len(sequences.shape) == 1:
                # 1D -> 3D 변환 (1, self._actual_h5_seq_len, num_features)
                sequences = sequences.reshape(1, self._actual_h5_seq_len, self.num_features)
            elif len(sequences.shape) == 2:
                # (self._actual_h5_seq_len, num_features) or (num_sequences, flattened)
                if sequences.shape[0] == self._actual_h5_seq_len and sequences.shape[1] == self.num_features:
                    # (seq_len, num_features) -> (1, seq_len, num_features)
                    sequences = sequences.reshape(1, self._actual_h5_seq_len, self.num_features)
                else:
                    # (num_sequences, flattened) -> (num_sequences, self._actual_h5_seq_len, num_features)
                    num_sequences = sequences.shape[0]
                    sequences = sequences.reshape(num_sequences, self._actual_h5_seq_len, self.num_features)
            # len(sequences.shape) == 3인 경우는 그대로 사용

            # Ensure sequences are of self.seq_len (the target configured length)
            if sequences.shape[1] < self.seq_len: # Pad if shorter (시퀀스 길이가 짧으면 패딩)
                padding_needed = self.seq_len - sequences.shape[1]
                # Pad with zeros along the sequence dimension (시퀀스 차원을 따라 0으로 패딩)
                padding_shape = (sequences.shape[0], padding_needed, sequences.shape[2])
                padding = np.zeros(padding_shape, dtype=sequences.dtype)
                sequences = np.concatenate((sequences, padding), axis=1)
            elif sequences.shape[1] > self.seq_len: # Truncate if longer (시퀀스 길이가 길면 잘라냄)
                sequences = sequences[:, :self.seq_len, :]

            return torch.from_numpy(sequences).float()

class BatchCollator:
    def __init__(self, max_sequences_per_batch=2000):
        self.max_sequences_per_batch = max_sequences_per_batch

    def __call__(self, batch_list):
        all_sequences = torch.cat(batch_list, dim=0)
        if len(all_sequences) > self.max_sequences_per_batch:
            indices = torch.randperm(len(all_sequences))[:self.max_sequences_per_batch]
            all_sequences = all_sequences[indices]
        return all_sequences

print("✅ Data classes defined (Fixed version with target_seq_len support)")
print("🔧 Import fix v2 applied successfully!")
print("📝 This version handles various HDF5 data structures and pads/truncates to target_seq_len")


## ⚙️ Training Configuration

In [ ]:
# Configuration
CONFIG = {
    'embedding_dim': 128,
    'hidden_dim': 256,
    'num_layers': 3,
    'dropout': 0.1,
    'mask_ratio': 0.15,

    'sequence_length': 120, # 시퀀스 길이 추가

    'batch_size': 1, # Further reduced batch size to 1 for memory
    'max_sequences': 500, # Reduced max_sequences to be conservative
    'max_epochs': 40,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,

    'max_batches_per_month': 30,
    'train_months': ['2024_09', '2024_10', '2024_11', '2024_12', '2025_01', '2025_02', '2025_03', '2025_04', '2025_05', '2025_06', '2025_07', '2025_08'],
    'val_months': ['2025_09'],
}

# GPU memory adjustment
if torch.cuda.is_available():
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🖥️ GPU Memory: {gpu_memory:.1f} GB")

    # Adjust max_sequences and batch_size based on sequence_length
    # 시퀀스 길이가 길어지면 메모리 사용량이 늘어나므로 조정
    base_mem_factor = (CONFIG['sequence_length'] / 60) # 60이 이전 기준 시퀀스 길이라고 가정

    # Removed automatic adjustment here, directly setting to lower values above due to OOM
    if gpu_memory < 12:
        # Original logic for small GPU
        # CONFIG['max_sequences'] = int(1000 / base_mem_factor)
        # CONFIG['batch_size'] = 2
        # CONFIG['max_batches_per_month'] = 20
        print("⚠️ Original config for limited GPU memory (factoring in sequence length) might be applied manually now.")
    elif gpu_memory > 25:
        # Original logic for large GPU
        # CONFIG['max_sequences'] = int(4000 / base_mem_factor)
        # CONFIG['batch_size'] = 8
        # CONFIG['max_batches_per_month'] = 100
        print("🚀 Original config for high-end GPU (factoring in sequence length) is not applied due to current settings.")

    # Add a check to ensure current manually set batch_size is not too high
    if CONFIG['batch_size'] > 1 and gpu_memory < 18: # Example heuristic
        print("⚠️ Manually adjusted batch_size may still be too high for this GPU. Consider further reduction.")

print("📋 Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")


## 📁 Data Setup

In [ ]:
# Check if classes are defined
print(f"Is PreprocessedDataset defined? {'PreprocessedDataset' in globals()}")
print(f"Is BatchCollator defined? {'BatchCollator' in globals()}")
print(f"Is MaskedAutoEncoder defined? {'MaskedAutoEncoder' in globals()}")

# Now try creating the datasets and dataloaders again
print("📊 Creating datasets...")

try:
    train_dataset = PreprocessedDataset(
        DATA_PATH,
        CONFIG['train_months'],
        CONFIG['max_batches_per_month'],
        configured_seq_len=CONFIG['sequence_length'] # Pass configured_seq_len (구성된 시퀀스 길이 전달)
    )

    val_dataset = PreprocessedDataset(
        DATA_PATH,
        CONFIG['val_months'],
        CONFIG['max_batches_per_month'],
        configured_seq_len=CONFIG['sequence_length'] # Pass configured_seq_len (구성된 시퀀스 길이 전달)
    )

    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        collate_fn=BatchCollator(CONFIG['max_sequences']),
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        collate_fn=BatchCollator(CONFIG['max_sequences']),
        num_workers=2,
        pin_memory=True
    )

    print(f"📈 Training batches: {len(train_loader)}")
    print(f"📉 Validation batches: {len(val_loader)}")
    print(f"🔢 Data shape: {train_dataset.seq_len} x {train_dataset.num_features}")
    print("✅ Data loaders created successfully.")

except Exception as e:
    print(f"❌ Error creating datasets or dataloaders: {e}")


## 🧠 Model Setup

In [ ]:
# Create model
print("🧠 Creating model...")

model = MaskedAutoEncoder(
    input_dim=train_dataset.num_features,
    embedding_dim=CONFIG['embedding_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    seq_len=CONFIG['sequence_length'], # Use CONFIG['sequence_length'] (설정된 시퀀스 길이 사용)
    num_layers=CONFIG['num_layers'],
    dropout=CONFIG['dropout'],
    mask_ratio=CONFIG['mask_ratio']
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1e6:.1f} MB")

# Create optimizer and scheduler
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    betas=(0.9, 0.95)
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['max_epochs'],
    eta_min=CONFIG['learning_rate'] * 0.01
)

print("✅ Model, optimizer, and scheduler created")


## 🚀 Training

In [ ]:
import time # Add this import statement
import torch.cuda.amp as amp # Import for mixed precision training

# Training loop
print("🚀 Starting training...")

# Clear CUDA cache to free up memory before starting training
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("✅ CUDA cache cleared.")

history = {'train_loss': [], 'val_loss': [], 'learning_rate': []}
best_val_loss = float('inf')
start_epoch = 0
start_time = time.time()

# Initialize GradScaler for mixed precision training
scaler = amp.GradScaler()
print("✅ Initialized GradScaler for mixed precision training.")

# --- Check for existing checkpoints to resume training ---
checkpoint_dir = Path(DATA_PATH)

# Prioritize best_model.pt if it exists and is newer or better
best_model_path = checkpoint_dir / 'best_model.pt'
latest_checkpoint_path = None
latest_epoch = -1

# Find the latest checkpoint_epoch_X.pt
for f in checkpoint_dir.glob('checkpoint_epoch_*.pt'):
    try:
        epoch_num = int(f.stem.split('_')[-1])
        if epoch_num > latest_epoch:
            latest_epoch = epoch_num
            latest_checkpoint_path = f
    except ValueError:
        continue

# Decide which checkpoint to load
resume_checkpoint_path = None
if best_model_path.exists():
    print(f"🔍 Found best model checkpoint at: {best_model_path}")
    resume_checkpoint_path = best_model_path

if latest_checkpoint_path and (not resume_checkpoint_path or latest_epoch > start_epoch): # Compare if latest_epoch is actually later than current start_epoch if best_model was loaded
    print(f"🔍 Found latest epoch checkpoint at: {latest_checkpoint_path}")
    resume_checkpoint_path = latest_checkpoint_path

if resume_checkpoint_path:
    print(f"🔄 Resuming training from checkpoint: {resume_checkpoint_path}")
    checkpoint = torch.load(resume_checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    history = checkpoint['history']
    best_val_loss = checkpoint['val_loss'] # Start with the validation loss of the loaded model
    # Restore scaler state if available (optional, but good practice)
    if 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
    print(f"✅ Resumed from Epoch {start_epoch-1}. Best Val Loss so far: {best_val_loss:.6f}")
else:
    print("🆕 No checkpoint found. Starting training from scratch.")


for epoch in range(start_epoch, CONFIG['max_epochs']):
    epoch_start = time.time()

    # Training
    model.train()
    train_loss = 0.0
    train_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['max_epochs']}")
    for batch in pbar:
        batch = batch.to(device)

        optimizer.zero_grad()

        # Use autocast for mixed precision training
        with amp.autocast():
            reconstruction, embedding, mask = model(batch)

            # Reconstruction loss (only on masked tokens)
            recon_loss = F.mse_loss(reconstruction[mask], batch[mask])

            # Regularization loss
            reg_loss = 0.01 * torch.norm(embedding, dim=1).mean()

            total_loss = recon_loss + reg_loss

        # Scale the loss and call backward()
        scaler.scale(total_loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) # Apply clip_grad_norm_ before scaler.step()

        # Optimizer step and scaler update
        scaler.step(optimizer)
        scaler.update()

        train_loss += total_loss.item()
        train_batches += 1

        pbar.set_postfix({
            'loss': f"{total_loss.item():.4f}",
            'recon': f"{recon_loss.item():.4f}",
            'reg': f"{reg_loss.item():.4f}",
            'avg': f"{train_loss/train_batches:.4f}"
        })

    train_loss /= train_batches

    # Validation
    model.eval()
    val_loss = 0.0
    val_batches = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            batch = batch.to(device)
            with amp.autocast(): # Use autocast for validation as well
                reconstruction, embedding, mask = model(batch)
                loss = F.mse_loss(reconstruction[mask], batch[mask])
            val_loss += loss.item()
            val_batches += 1

    val_loss /= val_batches

    # Update scheduler
    scheduler.step()

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])

    epoch_time = time.time() - epoch_start

    print(f"📊 Epoch {epoch+1} Results:")
    print(f"  Train Loss: {train_loss:.6f}")
    print(f"  Val Loss: {val_loss:.6f}")
    print(f"  Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"  Time: {epoch_time:.1f}s")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(), # Save scaler state
            'config': CONFIG,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'history': history
        }, f'{DATA_PATH}/best_model.pt')
        print(f"  💾 Best model saved (val_loss: {val_loss:.6f})")

    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(), # Save scaler state
            'config': CONFIG,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'history': history
        }, f'{DATA_PATH}/checkpoint_epoch_{epoch+1}.pt')
        print(f"  📁 Checkpoint saved: epoch_{epoch+1}.pt")

total_time = time.time() - start_time
print(f"🎉 Training completed!")
print(f"⏱️ Total time: {total_time/60:.1f} minutes")
print(f"🏆 Best validation loss: {best_val_loss:.6f}")


## 📈 Results Visualization

In [ ]:
# Plot training results
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss', color='blue')
plt.plot(history['val_loss'], label='Val Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(history['learning_rate'], color='green')
plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule')
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
plt.semilogy(history['train_loss'], label='Train Loss', color='blue')
plt.semilogy(history['val_loss'], label='Val Loss', color='red')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training Progress (Log Scale)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DATA_PATH}/training_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Save training history
with open(f'{DATA_PATH}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

print("📊 Training plots generated and history saved")

## 💾 Save Final Model to Google Drive

In [ ]:
# Save to Google Drive
drive_model_path = '/content/drive/MyDrive/ColabData/models/stockbot/autoencoder'
!mkdir -p "{drive_model_path}"

# Create timestamp for model versioning
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f'{timestamp}'

# Save model files
model_dir = f'{drive_model_path}/{model_name}'
!mkdir -p "{model_dir}"

print(f"💾 Saving model to: {model_dir}")

# Copy best model and artifacts
# Save 'best_model.pt' which is saved to DATA_PATH as determined by the training loop
!cp "{DATA_PATH}/best_model.pt" "{model_dir}/model.pt"
!cp "{DATA_PATH}/training_history.json" "{model_dir}/training_history.json"
!cp "{DATA_PATH}/training_results.png" "{model_dir}/training_results.png"

# Save model info
model_info = {
    'model_name': model_name,
    'timestamp': timestamp,
    'config': CONFIG,
    'best_val_loss': best_val_loss,
    'total_params': total_params,
    'training_time_minutes': total_time / 60,
    'data_shape': {
        'seq_len': CONFIG['sequence_length'], # Use CONFIG['sequence_length'] (설정된 시퀀스 길이 사용)
        'num_features': train_dataset.num_features
    }
}

with open(f'{model_dir}/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"✅ Model saved to: {model_dir}")
print(f"📁 Files saved:")
print(f"  - model.pt (best checkpoint)")
print(f"  - training_history.json")
print(f"  - training_results.png")
print(f"  - model_info.json")

# Display final summary
print(f"🎉 Training Summary:")
print(f"  Model: {model_name}")
print(f"  Best Val Loss: {best_val_loss:.6f}")
print(f"  Parameters: {total_params:,}")
print(f"  Training Time: {total_time/60:.1f} minutes")
print(f"  Embedding Dim: {CONFIG['embedding_dim']}")
print(f"  Data Shape: {CONFIG['sequence_length']} x {train_dataset.num_features}") # Use CONFIG['sequence_length'] (설정된 시퀀스 길이 사용)
